In [ ]:
# Example 1: verify the primary public API surface exists.
import click_repl

expected = ['register_repl', 'repl', 'exit']
found = {name: hasattr(click_repl, name) for name in expected}
for name, present in found.items():
    print(f"click_repl.{name}: {'OK' if present else 'MISSING'}")

assert callable(click_repl.register_repl), 'register_repl must be callable'
assert callable(click_repl.repl), 'repl must be callable'


In [ ]:
# Example 2: register_repl attaches a 'repl' subcommand to a click.Group.
import click
from click_repl import register_repl

@click.group()
def cli():
    pass

@cli.command()
def hello():
    click.echo('hello')

register_repl(cli)

subcommands = list(cli.commands.keys())
print(f"subcommands: {subcommands}")
assert 'repl' in subcommands, 'register_repl should add a repl subcommand'
assert 'hello' in subcommands


In [ ]:
# Example 3: register_repl supports a custom subcommand name.
import click
from click_repl import register_repl

@click.group()
def cli():
    pass

register_repl(cli, name='shell')

assert 'shell' in cli.commands, 'custom name should register under that key'
assert 'repl' not in cli.commands, 'default name should not appear when custom given'
print(f"registered under custom name: shell -> {cli.commands['shell']}")


In [ ]:
# Example 4: the ExitReplException is raised by click_repl.exit().
import click_repl
from click_repl import exit as repl_exit
from click_repl.exceptions import ExitReplException

raised = False
try:
    repl_exit()
except ExitReplException:
    raised = True
except SystemExit:
    # some versions raise SystemExit instead
    raised = True

assert raised, 'click_repl.exit() should raise ExitReplException (or SystemExit)'
print('exit() correctly raises an exit signal')


In [ ]:
# Example 5: ClickCompleter from click_repl exposes a get_completions iterator.
import click
from click_repl._completer import ClickCompleter
from prompt_toolkit.document import Document

@click.group()
def cli():
    pass

@cli.command()
def alpha():
    pass

@cli.command()
def beta():
    pass

ctx = click.Context(cli)
completer = ClickCompleter(cli, ctx)

doc = Document(text='al', cursor_position=2)
completions = list(completer.get_completions(doc, None))
texts = [c.text for c in completions]
print(f"completions for 'al': {texts}")
assert any('alpha' in t for t in texts), 'should suggest alpha for prefix al'
